# 02a - MNAR-Resolved & Leakage-Safe Retrain (Experiment 2 only)

**Purpose:** Section 4.4 of the Final Project Report documented two findings — (1) an MNAR bias
introduced by truncating raw text at a fixed 2,000-character threshold, which disproportionately
affected long-form ham messages and Tamil/Malay text, and (2) template-skeleton leakage, where
~6.6-6.8% of validation/test messages shared the same slot-filled template shell as a training
message despite having zero exact-text overlap.

This notebook resolves both issues and retrains **only Experiment 2** (lr=2e-5, batch=32 — the
original winning configuration) to produce a leakage-corrected test F1 for comparison against the
originally reported 0.9929.

This notebook reuses the exact raw dataset paths and cleaning logic from
`02_data-preparation-preprocessing-eda.ipynb` — only the truncation step and the split strategy
change; everything else (loading, schema, per-language balancing) is identical so the two runs
stay comparable.


## 1. Imports & Config

In [5]:
import os
import json
import warnings
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix

warnings.filterwarnings("ignore")

# output directories
BASE_DIR    = "/kaggle/working"
DATA_DIR    = os.path.join(BASE_DIR, "data_v2_leakage_safe")   # separate from the original NB02 outputs
os.makedirs(DATA_DIR, exist_ok=True)

# dataset paths — identical to 02_data-preparation-preprocessing-eda.ipynb
# (update these if your Kaggle dataset mount paths differ)
RAW_SYNTH   = "/kaggle/input/datasets/bhoovika/scamscene-raw-dataset/synthetic_dataset.csv"
RAW_PHISH   = "/kaggle/input/datasets/naserabdullahalam/phishing-email-dataset/phishing_email.csv"
RAW_UCI     = "/kaggle/input/datasets/bhoovika/scamscene-raw-dataset/SMSSpamCollection"
RAW_JOBS    = "/kaggle/input/datasets/shivamb/real-or-fake-fake-jobposting-prediction/fake_job_postings.csv"
RAW_NUS_EN  = "/kaggle/input/datasets/rtatman/the-national-university-of-singapore-sms-corpus/smsCorpus_en_2015.03.09_all.json"
RAW_NUS_ZH  = "/kaggle/input/datasets/rtatman/the-national-university-of-singapore-sms-corpus/smsCorpus_zh_2015.03.09.json"
RAW_WIKI_MS = "/kaggle/input/datasets/bhoovika/scamscene-raw-dataset/wiki_ms.csv"
RAW_WIKI_TA = "/kaggle/input/datasets/bhoovika/scamscene-raw-dataset/wiki_ta.csv"
RAW_WIKI_ZH = "/kaggle/input/datasets/bhoovika/scamscene-raw-dataset/wiki_zh.csv"

VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_STATE = 42

print("Imports done")
print("DATA_DIR:", DATA_DIR)


Imports done
DATA_DIR: /kaggle/working/data_v2_leakage_safe


## 2. Load & Merge Raw Data (identical to NB02)

In [6]:
scam_frames = []

df_synth = pd.read_csv(RAW_SYNTH, encoding="utf-8-sig")
df_synth = df_synth[["text", "label", "language", "scam_type"]].copy()
df_synth["source"] = "synthetic_spf2025"
scam_frames.append(df_synth)
print(f"{'Synthetic scam':<25}: {len(df_synth):>7,} rows")

df_phishing = pd.read_csv(RAW_PHISH, usecols=["text_combined", "label"], on_bad_lines="skip")
df_phishing.columns = ["text", "label"]
df_phishing["language"] = "en"
df_phishing["source"] = "phishing_email"
df_phishing_scam = df_phishing[df_phishing["label"] == 1].copy()
df_phishing_ham  = df_phishing[df_phishing["label"] == 0].copy()
scam_frames.append(df_phishing_scam)
print(f"{'Phishing Email scam':<25}: {len(df_phishing_scam):>7,} rows")

df_uci = pd.read_csv(RAW_UCI, sep="\t", header=None, names=["label_str", "text"], on_bad_lines="skip")
df_uci_spam = df_uci[df_uci["label_str"] == "spam"].copy()
df_uci_spam["label"] = 1
df_uci_spam["language"] = "en"
df_uci_spam["source"] = "uci_sms_spam"
df_uci_spam = df_uci_spam[["text", "label", "language", "source"]]
scam_frames.append(df_uci_spam)
print(f"{'UCI SMS scam':<25}: {len(df_uci_spam):>7,} rows")

df_jobs = pd.read_csv(RAW_JOBS, on_bad_lines="skip")
df_jobs["text"] = (df_jobs["title"].fillna("") + " " + df_jobs["description"].fillna("")).str.strip()
df_jobs_scam = df_jobs[df_jobs["fraudulent"] == 1].copy()
df_jobs_scam["label"] = 1
df_jobs_scam["language"] = "en"
df_jobs_scam["source"] = "job_postings_fake"
df_jobs_scam = df_jobs_scam[["text", "label", "language", "source"]]
scam_frames.append(df_jobs_scam)
print(f"{'Fake job postings scam':<25}: {len(df_jobs_scam):>7,} rows")

df_scam = pd.concat(scam_frames, ignore_index=True)
print(f"\nTotal scam rows (before cleaning): {len(df_scam):,}")


Synthetic scam           :  24,383 rows
Phishing Email scam      :  42,891 rows
UCI SMS scam             :     747 rows
Fake job postings scam   :     866 rows

Total scam rows (before cleaning): 68,887


In [7]:
ham_frames = []

ham_frames.append(df_phishing_ham)
print(f"{'Phishing Email ham':<25}: {len(df_phishing_ham):>7,} rows")

df_uci_ham = df_uci[df_uci["label_str"] == "ham"].copy()
df_uci_ham["label"] = 0
df_uci_ham["language"] = "en"
df_uci_ham["source"] = "uci_sms_ham"
df_uci_ham = df_uci_ham[["text", "label", "language", "source"]]
ham_frames.append(df_uci_ham)
print(f"{'UCI SMS ham':<25}: {len(df_uci_ham):>7,} rows")

NUS_FILES = [(RAW_NUS_EN, "singlish"), (RAW_NUS_ZH, "zh")]
for fpath, lang in NUS_FILES:
    with open(fpath, encoding="utf-8") as f:
        raw_json = json.load(f)
    messages = raw_json["smsCorpus"]["message"]
    texts = [m["text"]["$"] for m in messages if "$" in m.get("text", {})]
    df_nus = pd.DataFrame({"text": texts, "label": 0, "language": lang, "source": "nus_sms"})
    ham_frames.append(df_nus)
    print(f"{f'NUS SMS ham ({lang})':<25}: {len(df_nus):>7,} rows")

df_jobs_ham = df_jobs[df_jobs["fraudulent"] == 0].copy()
df_jobs_ham["text"] = (df_jobs_ham["title"].fillna("") + " " + df_jobs_ham["description"].fillna("")).str.strip()
df_jobs_ham["label"] = 0
df_jobs_ham["language"] = "en"
df_jobs_ham["source"] = "job_postings_real"
df_jobs_ham = df_jobs_ham[["text", "label", "language", "source"]]
ham_frames.append(df_jobs_ham)
print(f"{'Real job postings ham':<25}: {len(df_jobs_ham):>7,} rows")

WIKI_FILES = [(RAW_WIKI_MS, "ms"), (RAW_WIKI_TA, "ta"), (RAW_WIKI_ZH, "zh")]
for fpath, lang in WIKI_FILES:
    df_wiki = pd.read_csv(fpath, usecols=["text"], on_bad_lines="skip")
    df_wiki["label"] = 0
    df_wiki["language"] = lang
    df_wiki["source"] = f"wikipedia_{lang}"
    ham_frames.append(df_wiki)
    print(f"{f'Wikipedia ham ({lang})':<25}: {len(df_wiki):>7,} rows")

df_ham = pd.concat(ham_frames, ignore_index=True)
print(f"\nTotal ham rows (before cleaning): {len(df_ham):,}")


Phishing Email ham       :  39,595 rows
UCI SMS ham              :   4,825 rows
NUS SMS ham (singlish)   :  55,835 rows
NUS SMS ham (zh)         :  31,460 rows
Real job postings ham    :  17,014 rows
Wikipedia ham (ms)       :   5,000 rows
Wikipedia ham (ta)       :   5,000 rows
Wikipedia ham (zh)       :   5,000 rows

Total ham rows (before cleaning): 163,729


In [8]:
df = pd.concat([df_scam, df_ham], ignore_index=True)
print(f"Raw merged: {df.shape}")

df["text"]     = df["text"].astype(str)
df["label"]    = df["label"].astype(int)
df["language"] = df["language"].astype(str)
df["source"]   = df["source"].astype(str)

def assign_scam_type(row):
    if pd.notna(row["scam_type"]):
        synthetic_map = {
            "phishing": "scam_phishing", "rental": "scam_rental", "investment": "scam_investment",
            "charity": "scam_charity", "parcel_delivery": "scam_parcel_delivery", "fake_friend": "scam_fake_friend",
            "prize": "scam_prize", "bank_impersonation": "scam_bank_impersonation", "loan": "scam_loan",
            "ecommerce": "scam_ecommerce", "government_impersonation": "scam_government_impersonation",
            "job_scam": "scam_job",
        }
        return synthetic_map.get(row["scam_type"], row["scam_type"])
    s = str(row["source"]).lower()
    if row["label"] == 0:
        if "nus" in s:      return "ham_sms"
        if "wiki" in s:     return "ham_text"
        if "job" in s:      return "ham_job"
        if "phishing" in s: return "ham_email"
        if "uci" in s:      return "ham_sms"
        return "ham_other"
    if "phishing" in s: return "scam_phishing"
    if "job" in s:      return "scam_job"
    if "uci" in s:      return "scam_sms_spam"
    return "scam_other"

df["scam_type"] = df.apply(assign_scam_type, axis=1)
print(f"Schema standardised. Shape: {df.shape}")


Raw merged: (232616, 5)
Schema standardised. Shape: (232616, 5)


## 3. Cleaning — MNAR-Resolved Version

**Change from NB02:** the original pipeline truncated raw text at a fixed 2,000-character
threshold before tokenisation. Section 4.4 of the report found this truncation was non-random —
45.6% of `ham_text` and 24.9% of Tamil messages were truncated versus 0% of eleven of twelve
synthetic scam categories — creating an MNAR bias where message length could act as a spurious
signal correlated with the class label and language.

Since the tokenizer already caps every message uniformly at 128 tokens regardless of raw
character length (Section 3.7 of the report), the character-level truncation step is **removed
here**. Truncation is deferred entirely to tokenisation in Section 8 below, which is applied
identically to every row irrespective of class or language — removing the bias at its source
rather than patching around it after the fact. All other cleaning steps (null removal, short-text
removal, deduplication) are unchanged from NB02.


In [9]:
before = len(df)
df = df.dropna(subset=["text"])
print(f"{'Dropped nulls':<40}: {before - len(df):>7,} rows")

df["text"] = df["text"].str.strip()

before = len(df)
df = df[df["text"].str.lower() != "nan"]
print(f"{'Dropped literal nan strings':<40}: {before - len(df):>7,} rows")

before = len(df)
df = df[df["text"].str.len() >= 5]
print(f"{'Dropped short texts (<5 chars)':<40}: {before - len(df):>7,} rows")

# NOTE: no character-level truncation here (MNAR fix) — truncation deferred to
# tokenisation (Section 8), applied uniformly at max_length=128 tokens for every row.
print(f"{'Character truncation':<40}: SKIPPED (deferred to tokenizer, uniform across all rows)")

before = len(df)
df = df.drop_duplicates(subset=["text", "language"]).reset_index(drop=True)
print(f"{'Dropped (text, language) duplicates':<40}: {before - len(df):>7,} rows")

before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"{'Dropped cross-language duplicates':<40}: {before - len(df):>7,} rows")

df["_norm"] = df["text"].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.strip()
before = len(df)
df = df.drop_duplicates(subset=["_norm"]).reset_index(drop=True)
df.drop(columns=["_norm"], inplace=True)
print(f"{'Dropped near-duplicates (normalised)':<40}: {before - len(df):>7,} rows")

before = len(df)
df = df[df["text"].str.replace(r"[^\w\s]", "", regex=True).str.strip().str.len() >= 5]
print(f"{'Dropped punctuation-only rows':<40}: {before - len(df):>7,} rows")

print(f"\n{'Dataset shape':<40}: {str(df.shape):>7}")


Dropped nulls                           :       0 rows
Dropped literal nan strings             :       0 rows
Dropped short texts (<5 chars)          :   6,944 rows
Character truncation                    : SKIPPED (deferred to tokenizer, uniform across all rows)
Dropped (text, language) duplicates     :   9,579 rows
Dropped cross-language duplicates       :   3,476 rows
Dropped near-duplicates (normalised)    :   1,496 rows
Dropped punctuation-only rows           :   1,482 rows

Dataset shape                           : (209639, 5)


## 4. Per-Language Class Balancing (identical to NB02)

In [10]:
scam_before = (df["label"] == 1).sum()
ham_before  = (df["label"] == 0).sum()
print(f"Before per-lang undersample: scam={scam_before:,}  ham={ham_before:,}\n")

balanced = []
lang_stats = []
for lang in df["language"].unique():
    sub = df[df["language"] == lang]
    scam_n = (sub["label"] == 1).sum()
    ham_n  = (sub["label"] == 0).sum()
    df_s = sub[sub["label"] == 1]
    df_h = sub[sub["label"] == 0]
    if ham_n > scam_n:
        df_h = df_h.sample(n=scam_n, random_state=RANDOM_STATE)
        ham_kept = scam_n
        ham_display = f"{ham_n:,} -> {scam_n:,}"
    else:
        ham_kept = ham_n
        ham_display = f"{ham_n:,}"
    balanced.extend([df_s, df_h])
    total = scam_n + ham_kept
    lang_stats.append({"language": lang, "ham_perlang": ham_display, "scam_perlang": scam_n,
                        "total_perlang": total, "balance_perlang": round(scam_n / total, 3)})

df_balanced = pd.concat(balanced, ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

scam_after = (df_balanced["label"] == 1).sum()
ham_after  = (df_balanced["label"] == 0).sum()
print(f"After per-lang undersample:  scam={scam_after:,}  ham={ham_after:,}\n")
print(pd.DataFrame(lang_stats).set_index("language").to_string())
print(f"\nBalanced shape: {df_balanced.shape}")


Before per-lang undersample: scam=68,512  ham=141,127

After per-lang undersample:  scam=68,512  ham=68,512

               ham_perlang  scam_perlang  total_perlang  balance_perlang
language                                                                
zh         29,333 -> 4,962          4962           9924              0.5
en        58,708 -> 49,140         49140          98280              0.5
ta          4,997 -> 4,637          4637           9274              0.5
ms          4,974 -> 4,773          4773           9546              0.5
singlish   43,115 -> 5,000          5000          10000              0.5

Balanced shape: (137024, 5)


## 5. Leakage-Safe Split — Grouped by Template Skeleton

**Change from NB02:** the original split (NB02) stratified only on `label + language`, which
allowed synthetic messages sharing the same slot-filled template to land on both sides of the
split — Section 4.4 found ~6.6-6.8% skeleton overlap between train and val/test despite zero
exact-text overlap. This split instead groups rows by their template skeleton (slot values
masked out) using `GroupShuffleSplit`, guaranteeing no template shell appears in more than one
split. Because grouping — not stratification — now drives the split, label/language balance per
split is checked below rather than guaranteed; if it drifts meaningfully, note that transparently
in the writeup rather than adjusting the split to force it back.


In [11]:
def template_skeleton(text):
    """Strip slot-filled values, leaving only the template shell."""
    t = str(text)
    t = re.sub(r'\d+', '<NUM>', t)
    t = re.sub(r'\$?\d+[,.]?\d*', '<AMOUNT>', t)
    t = re.sub(r'https?://\S+', '<URL>', t)
    t = re.sub(r'\b[A-Z][a-z]+\b', '<NAME>', t)
    return t.lower().strip()

df_balanced["_skeleton"] = df_balanced["text"].apply(template_skeleton)

# train vs (val+test), grouped by skeleton
gss1 = GroupShuffleSplit(n_splits=1, test_size=(VAL_RATIO + TEST_RATIO), random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(df_balanced, groups=df_balanced["_skeleton"]))
train_df = df_balanced.iloc[train_idx].reset_index(drop=True)
temp_df  = df_balanced.iloc[temp_idx].reset_index(drop=True)

# val vs test, grouped by skeleton (so no leakage between val/test either)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_STATE)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["_skeleton"]))
val_df  = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

for d in [train_df, val_df, test_df]:
    d.drop(columns=["_skeleton"], inplace=True)

total = len(df_balanced)
print(f"Train: {len(train_df):,} ({len(train_df)/total*100:.1f}%)")
print(f"Val:   {len(val_df):,}   ({len(val_df)/total*100:.1f}%)")
print(f"Test:  {len(test_df):,}  ({len(test_df)/total*100:.1f}%)")

print("\nLabel balance per split (check for drift vs. the original ~50/50 by design):")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    c = d["label"].value_counts(normalize=True).round(3)
    print(f"  {name}: scam={c.get(1,0):.3f}  ham={c.get(0,0):.3f}")

print("\nLanguage balance per split:")
for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"  {name}:")
    print(d["language"].value_counts(normalize=True).round(3).to_string().replace("\n", "\n    "))


Train: 95,675 (69.8%)
Val:   20,849   (15.2%)
Test:  20,500  (15.0%)

Label balance per split (check for drift vs. the original ~50/50 by design):
  Train: scam=0.503  ham=0.497
  Val: scam=0.494  ham=0.506
  Test: scam=0.493  ham=0.507

Language balance per split:
  Train:
language
    en          0.716
    singlish    0.074
    zh          0.073
    ms          0.069
    ta          0.068
  Val:
language
    en          0.729
    ms          0.070
    zh          0.069
    singlish    0.069
    ta          0.064
  Test:
language
    en          0.714
    singlish    0.074
    zh          0.072
    ms          0.071
    ta          0.070


## 6. Split Integrity & Leakage Verification (should now show zero on both checks)

In [12]:
print("SPLIT INTEGRITY & LEAKAGE CHECKS (post-fix)")

train_texts = set(train_df["text"].astype(str))
val_texts   = set(val_df["text"].astype(str))
test_texts  = set(test_df["text"].astype(str))

print("1. Exact text overlap")
print(f"   Train ∩ Val  : {len(train_texts & val_texts):,}")
print(f"   Train ∩ Test : {len(train_texts & test_texts):,}")
print(f"   Val   ∩ Test : {len(val_texts & test_texts):,}")

train_skel = set(train_df["text"].astype(str).apply(template_skeleton))
val_skel   = set(val_df["text"].astype(str).apply(template_skeleton))
test_skel  = set(test_df["text"].astype(str).apply(template_skeleton))

print("\n2. Template-skeleton overlap (this is the check that used to show ~6.6-6.8% overlap)")
print(f"   Train ∩ Test skeletons : {len(train_skel & test_skel):,} / {len(test_skel):,} test skeletons")
print(f"   Train ∩ Val  skeletons : {len(train_skel & val_skel):,} / {len(val_skel):,} val skeletons")

assert len(train_skel & test_skel) == 0, "Skeleton leakage still present between train/test!"
assert len(train_skel & val_skel) == 0,  "Skeleton leakage still present between train/val!"
print("\n   Confirmed: zero exact-text AND zero template-skeleton overlap across all splits.")


SPLIT INTEGRITY & LEAKAGE CHECKS (post-fix)
1. Exact text overlap
   Train ∩ Val  : 0
   Train ∩ Test : 0
   Val   ∩ Test : 0

2. Template-skeleton overlap (this is the check that used to show ~6.6-6.8% overlap)
   Train ∩ Test skeletons : 0 / 19,054 test skeletons
   Train ∩ Val  skeletons : 0 / 19,054 val skeletons

   Confirmed: zero exact-text AND zero template-skeleton overlap across all splits.


## 7. Save Leakage-Safe Splits

In [13]:
df_balanced.to_csv(os.path.join(DATA_DIR, "scamsense_full_dataset_v2.csv"), index=False, encoding="utf-8-sig")
train_df.to_csv(os.path.join(DATA_DIR, "train.csv"), index=False, encoding="utf-8-sig")
val_df.to_csv(os.path.join(DATA_DIR, "val.csv"),   index=False, encoding="utf-8-sig")
test_df.to_csv(os.path.join(DATA_DIR, "test.csv"),  index=False, encoding="utf-8-sig")

for fname in ["scamsense_full_dataset_v2.csv", "train.csv", "val.csv", "test.csv"]:
    p = os.path.join(DATA_DIR, fname)
    print(f"  saved {fname}  ({os.path.getsize(p)/1e6:.1f} MB)")


  saved scamsense_full_dataset_v2.csv  (402.7 MB)
  saved train.csv  (139.9 MB)
  saved val.csv  (30.8 MB)
  saved test.csv  (29.9 MB)


## 8. Tokenisation (XLM-RoBERTa, max_length=128)

In [14]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

MODEL_CHECKPOINT = "xlm-roberta-base"   # same base checkpoint as the original Experiment 2 run
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def to_hf_dataset(df_):
    ds = Dataset.from_pandas(df_[["text", "label"]].reset_index(drop=True))
    return ds

train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds   = val_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
val_ds   = val_ds.rename_column("label", "labels")
test_ds  = test_ds.rename_column("label", "labels")

cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols)
val_ds.set_format(type="torch", columns=cols)
test_ds.set_format(type="torch", columns=cols)

print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/95675 [00:00<?, ? examples/s]

Map:   0%|          | 0/20849 [00:00<?, ? examples/s]

Map:   0%|          | 0/20500 [00:00<?, ? examples/s]

Train: 95,675  Val: 20,849  Test: 20,500


## 9. Fine-Tune XLM-RoBERTa — Experiment 2 Configuration Only

Matches the original Experiment 2 setup from Section 3.7 of the report: learning rate 2e-5,
batch size 32, AdamW, weight decay 0.01, linear warmup over 10% of steps, FP16 mixed precision,
gradient norm clipping at 1.0, early stopping on validation F1 with patience of 1 evaluation
epoch. Only this configuration is retrained here — Experiments 1, 3, WeightedRandomSampler, text
normalisation, and the mBERT baseline are unaffected by MNAR/leakage fixes and are not rerun.


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall":    recall_score(labels, preds, average="weighted"),
        "auc":       roc_auc_score(labels, probs[:, 1]),
    }

model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

training_args = TrainingArguments(
    output_dir="/kaggle/working/exp2_leakage_safe",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,               # capped by early stopping, same as original Exp2
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc
1,0.101731,0.105771,0.985611,0.985609,0.985701,0.985611,0.998639
2,0.090742,0.073663,0.991079,0.991079,0.991085,0.991079,0.999370
3,0.060508,0.067249,0.991606,0.991607,0.991625,0.991606,0.999571
4,0.040991,0.112409,0.989208,0.989206,0.989363,0.989208,0.998975


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=5980, training_loss=0.14395636861340258, metrics={'train_runtime': 6317.4105, 'train_samples_per_second': 151.447, 'train_steps_per_second': 2.366, 'total_flos': 2.5173150221568e+16, 'train_loss': 0.14395636861340258, 'epoch': 4.0})

## 10. Evaluate on the Leakage-Safe Test Set

In [16]:
test_results = trainer.evaluate(test_ds)
print("Leakage-safe test set results:")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k:<25}: {v:.4f}")

preds_output = trainer.predict(test_ds)
probs = torch.softmax(torch.tensor(preds_output.predictions), dim=-1).numpy()
preds = probs.argmax(axis=-1)
labels = preds_output.label_ids

print("\nConfusion matrix (rows=true, cols=pred, order=[ham, scam]):")
print(confusion_matrix(labels, preds))


Leakage-safe test set results:
  eval_loss                : 0.0851
  eval_accuracy            : 0.9901
  eval_f1                  : 0.9901
  eval_precision           : 0.9901
  eval_recall              : 0.9901
  eval_auc                 : 0.9995
  eval_runtime             : 107.9761
  eval_samples_per_second  : 189.8570
  eval_steps_per_second    : 2.9730
  epoch                    : 4.0000

Confusion matrix (rows=true, cols=pred, order=[ham, scam]):
[[10246   140]
 [   63 10051]]


## 11. Summary — Leakage-Corrected F1 vs. Originally Reported F1

In [17]:
original_test_f1 = 0.9929   # Experiment 2, as reported in Section 4.1.1 of the report

new_test_f1 = test_results.get("eval_f1")
delta = new_test_f1 - original_test_f1

print("EXPERIMENT 2 — MNAR & LEAKAGE FIX COMPARISON")
print(f"  Originally reported test F1 (char-truncated, label+lang-stratified split): {original_test_f1:.4f}")
print(f"  Leakage-safe test F1 (tokenizer-only truncation, skeleton-grouped split):  {new_test_f1:.4f}")
print(f"  Delta: {delta:+.4f}")
print()
if abs(delta) < 0.01:
    print("  Interpretation: F1 is stable under the leakage-safe split — the original near-ceiling")
    print("  score does not appear to be substantially inflated by template-level leakage or MNAR bias.")
else:
    print("  Interpretation: F1 shifted meaningfully under the leakage-safe split — this quantifies")
    print("  the extent to which the original score was affected by template-level leakage and/or")
    print("  the MNAR truncation bias documented in Section 4.4.")


EXPERIMENT 2 — MNAR & LEAKAGE FIX COMPARISON
  Originally reported test F1 (char-truncated, label+lang-stratified split): 0.9929
  Leakage-safe test F1 (tokenizer-only truncation, skeleton-grouped split):  0.9901
  Delta: -0.0028

  Interpretation: F1 is stable under the leakage-safe split — the original near-ceiling
  score does not appear to be substantially inflated by template-level leakage or MNAR bias.
